# House Price Prediction — Week 1 Project
## XYlofy AI Internship

**Project Goal:** Build a regression model to predict house prices and identify the most influential features.

---

## Task 1: Data Loading & Exploration

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Set style for better-looking plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Step 1.1: Load the dataset
# If using Colab, run this cell first:
# !wget -q https://raw.githubusercontent.com/yasserh/Housing-Prices-Dataset/main/Housing.csv

# Load CSV file
df = pd.read_csv('Housing.csv')

print("✓ Dataset loaded successfully!")
print(f"Dataset shape: {df.shape}")

In [ ]:
# Step 1.2: Display first 10 rows
print("First 10 rows of the dataset:")
print(df.head(10))

In [ ]:
# Step 1.3: Check dataset info
print("\n=== Dataset Information ===")
print(f"Total Rows: {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}")
print("\nColumn names and data types:")
print(df.dtypes)
print("\nDataset Info:")
df.info()

In [ ]:
# Step 1.4: Identify target and features
print("\n=== Target & Features ===")
print(f"Target Variable (to predict): price")
print(f"\nFeature Columns (predictors):")
for col in df.columns:
    if col.lower() != 'price':
        print(f"  - {col}")

In [ ]:
# Step 1.5: Check for missing values
print("\n=== Missing Values ===")
missing_summary = pd.DataFrame({
    'Column': df.columns,
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2)
})
print(missing_summary[missing_summary['Missing Count'] > 0])
print(f"\nTotal missing values in dataset: {df.isnull().sum().sum()}")

In [ ]:
# Step 1.6: Basic statistics
print("\n=== Statistical Summary ===")
print(df.describe())

---
## Task 2: Data Cleaning

In [ ]:
# Step 2.1: Handle missing values
print("=== Handling Missing Values ===")

# Check which columns have missing values
missing_cols = df.columns[df.isnull().any()].tolist()

if len(missing_cols) > 0:
    print(f"Columns with missing values: {missing_cols}")
    
    # For numeric columns: fill with median
    # For categorical columns: fill with mode
    for col in missing_cols:
        if df[col].dtype in ['float64', 'int64']:
            df[col].fillna(df[col].median(), inplace=True)
            print(f"  ✓ {col}: Filled with median")
        else:
            df[col].fillna(df[col].mode()[0], inplace=True)
            print(f"  ✓ {col}: Filled with mode")
else:
    print("✓ No missing values found!")

print(f"\nMissing values after cleaning: {df.isnull().sum().sum()}")

In [ ]:
# Step 2.2: Remove duplicate rows
print("\n=== Removing Duplicates ===")
print(f"Duplicate rows before removal: {df.duplicated().sum()}")

df_clean = df.drop_duplicates().reset_index(drop=True)

print(f"Duplicate rows after removal: {df_clean.duplicated().sum()}")
print(f"Dataset shape after removing duplicates: {df_clean.shape}")

df = df_clean

In [ ]:
# Step 2.3: Identify and encode categorical columns
print("\n=== Identifying Categorical Columns ===")

# Find columns with object or mixed data types
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical columns found: {categorical_cols}")

# Also check for columns with few unique values (likely categorical)
print("\nColumns with few unique values (possible categorical):")
for col in df.columns:
    if df[col].nunique() <= 10:
        print(f"  {col}: {df[col].unique()}")

In [ ]:
# Step 2.4: One-hot encode categorical variables
print("\n=== Encoding Categorical Variables ===")

# Apply one-hot encoding
df_encoded = pd.get_dummies(df, drop_first=True)

print(f"Shape before encoding: {df.shape}")
print(f"Shape after encoding: {df_encoded.shape}")
print(f"\nNew columns created:")
new_cols = set(df_encoded.columns) - set(df.columns)
for col in sorted(new_cols):
    print(f"  - {col}")

df = df_encoded

In [ ]:
# Step 2.5: Check cleaned dataset
print("\n=== Cleaned Dataset Summary ===")
print(df.head())
print(f"\nFinal dataset shape: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Data types:\n{df.dtypes.value_counts()}")

---
## Task 3: Model Building & Evaluation

In [ ]:
# Step 3.1: Prepare features and target
print("=== Preparing Data for Modeling ===")

# Identify target column (usually 'price')
target_col = [col for col in df.columns if col.lower() in ['price', 'target']]
if not target_col:
    print("⚠️ Warning: Could not find 'price' column. Checking column names...")
    print(df.columns.tolist())
    # Manually specify if needed:
    target_col = 'price'  # Change this if needed
else:
    target_col = target_col[0]

# Separate features and target
X = df.drop(columns=[target_col])
y = df[target_col]

print(f"✓ Target variable: {target_col}")
print(f"✓ Number of features: {X.shape[1]}")
print(f"✓ Number of samples: {X.shape[0]}")
print(f"\nFeature columns: {list(X.columns)}")

In [ ]:
# Step 3.2: Split data into train and test sets (80/20)
print("\n=== Train-Test Split ===")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Test set size: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\nTraining features shape: {X_train.shape}")
print(f"Test features shape: {X_test.shape}")

In [ ]:
# Step 3.3: Train Linear Regression Model
print("\n=== Training Linear Regression Model ===")

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Make predictions
y_train_pred_lr = lr_model.predict(X_train)
y_test_pred_lr = lr_model.predict(X_test)

print("✓ Linear Regression model trained!")

In [ ]:
# Step 3.4: Evaluate Linear Regression
print("\n=== Linear Regression Evaluation ===")

# Calculate metrics
lr_train_mae = mean_absolute_error(y_train, y_train_pred_lr)
lr_test_mae = mean_absolute_error(y_test, y_test_pred_lr)

lr_train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred_lr))
lr_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_lr))

lr_train_r2 = r2_score(y_train, y_train_pred_lr)
lr_test_r2 = r2_score(y_test, y_test_pred_lr)

# Display results
print(f"\nTraining Set:")
print(f"  MAE (Mean Absolute Error):   ${lr_train_mae:,.2f}")
print(f"  RMSE (Root Mean Squared Error): ${lr_train_rmse:,.2f}")
print(f"  R² Score:                    {lr_train_r2:.4f}")

print(f"\nTest Set:")
print(f"  MAE (Mean Absolute Error):   ${lr_test_mae:,.2f}")
print(f"  RMSE (Root Mean Squared Error): ${lr_test_rmse:,.2f}")
print(f"  R² Score:                    {lr_test_r2:.4f}")

In [ ]:
# Step 3.5: Train Random Forest Regressor
print("\n=== Training Random Forest Regressor ===")

rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Make predictions
y_train_pred_rf = rf_model.predict(X_train)
y_test_pred_rf = rf_model.predict(X_test)

print("✓ Random Forest model trained!")

In [ ]:
# Step 3.6: Evaluate Random Forest
print("\n=== Random Forest Evaluation ===")

# Calculate metrics
rf_train_mae = mean_absolute_error(y_train, y_train_pred_rf)
rf_test_mae = mean_absolute_error(y_test, y_test_pred_rf)

rf_train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred_rf))
rf_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_rf))

rf_train_r2 = r2_score(y_train, y_train_pred_rf)
rf_test_r2 = r2_score(y_test, y_test_pred_rf)

# Display results
print(f"\nTraining Set:")
print(f"  MAE (Mean Absolute Error):   ${rf_train_mae:,.2f}")
print(f"  RMSE (Root Mean Squared Error): ${rf_train_rmse:,.2f}")
print(f"  R² Score:                    {rf_train_r2:.4f}")

print(f"\nTest Set:")
print(f"  MAE (Mean Absolute Error):   ${rf_test_mae:,.2f}")
print(f"  RMSE (Root Mean Squared Error): ${rf_test_rmse:,.2f}")
print(f"  R² Score:                    {rf_test_r2:.4f}")

In [ ]:
# Step 3.7: Model Comparison
print("\n" + "="*60)
print("MODEL COMPARISON (TEST SET)")
print("="*60)

comparison_df = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'R² Score'],
    'Linear Regression': [f"${lr_test_mae:,.2f}", f"${lr_test_rmse:,.2f}", f"{lr_test_r2:.4f}"],
    'Random Forest': [f"${rf_test_mae:,.2f}", f"${rf_test_rmse:,.2f}", f"{rf_test_r2:.4f}"]
})

print(comparison_df.to_string(index=False))

# Determine best model
print(f"\n🏆 Best Model: ", end="")
if rf_test_r2 > lr_test_r2:
    print(f"Random Forest (R² = {rf_test_r2:.4f})")
    best_model = rf_model
    best_predictions = y_test_pred_rf
    best_model_name = "Random Forest"
else:
    print(f"Linear Regression (R² = {lr_test_r2:.4f})")
    best_model = lr_model
    best_predictions = y_test_pred_lr
    best_model_name = "Linear Regression"

print(f"\nInterpretation: The {best_model_name} explains {best_model_name == 'Random Forest' and rf_test_r2*100 or lr_test_r2*100:.2f}% of the price variance.")

---
## Task 4: Visualization (3+ Charts)

In [ ]:
# Create output directory for charts
import os
os.makedirs('charts', exist_ok=True)
print("✓ Created 'charts' folder for saving visualizations")

In [ ]:
# CHART 1: Histogram of House Prices
print("\n=== Chart 1: Distribution of House Prices ===")

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.hist(y, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
plt.xlabel('Price', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Distribution of House Prices', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)

# Add statistics
plt.axvline(y.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: ${y.mean():,.0f}')
plt.axvline(y.median(), color='green', linestyle='--', linewidth=2, label=f'Median: ${y.median():,.0f}')
plt.legend(fontsize=10)

# Add box plot for additional insight
plt.subplot(1, 2, 2)
plt.boxplot(y, vert=True)
plt.ylabel('Price', fontsize=12)
plt.title('Box Plot of House Prices', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('charts/chart_1_price_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Price Statistics:")
print(f"  Min: ${y.min():,.2f}")
print(f"  Max: ${y.max():,.2f}")
print(f"  Mean: ${y.mean():,.2f}")
print(f"  Median: ${y.median():,.2f}")
print(f"  Std Dev: ${y.std():,.2f}")
print("✓ Chart 1 saved to: charts/chart_1_price_distribution.png")

In [ ]:
# CHART 2: Correlation Heatmap
print("\n=== Chart 2: Correlation Heatmap ===")

# Calculate correlation with target variable
correlation = df.corr()[target_col].sort_values(ascending=False)

# Create heatmap of top 10 features + target
top_features = correlation.head(11).index.tolist()  # Top 10 + target
corr_matrix = df[top_features].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix: Top Features vs Price', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('charts/chart_2_correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nTop Features by Correlation with Price:")
print(correlation.head(11))
print("✓ Chart 2 saved to: charts/chart_2_correlation_heatmap.png")

In [ ]:
# CHART 3: Actual vs Predicted Prices (Scatter Plot)
print("\n=== Chart 3: Actual vs Predicted Prices ===")

plt.figure(figsize=(14, 6))

# Linear Regression
plt.subplot(1, 2, 1)
plt.scatter(y_test, y_test_pred_lr, alpha=0.6, color='steelblue', edgecolor='black')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2, label='Perfect Prediction')
plt.xlabel('Actual Price', fontsize=12)
plt.ylabel('Predicted Price', fontsize=12)
plt.title(f'Linear Regression: Actual vs Predicted\nR² = {lr_test_r2:.4f}', fontsize=12, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

# Random Forest
plt.subplot(1, 2, 2)
plt.scatter(y_test, y_test_pred_rf, alpha=0.6, color='darkgreen', edgecolor='black')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2, label='Perfect Prediction')
plt.xlabel('Actual Price', fontsize=12)
plt.ylabel('Predicted Price', fontsize=12)
plt.title(f'Random Forest: Actual vs Predicted\nR² = {rf_test_r2:.4f}', fontsize=12, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('charts/chart_3_actual_vs_predicted.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Chart 3 saved to: charts/chart_3_actual_vs_predicted.png")

In [ ]:
# BONUS CHART: Feature Importance (Random Forest)
print("\n=== Bonus Chart: Feature Importance (Random Forest) ===")

feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False).head(15)

plt.figure(figsize=(12, 6))
plt.barh(feature_importance['Feature'], feature_importance['Importance'], color='forestgreen', edgecolor='black')
plt.xlabel('Importance Score', fontsize=12)
plt.title('Top 15 Most Important Features (Random Forest)', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('charts/chart_4_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10).to_string(index=False))
print("\n✓ Bonus chart saved to: charts/chart_4_feature_importance.png")

---
## Task 5: Insights & Summary

In [ ]:
# INSIGHTS & SUMMARY
print("\n" + "="*70)
print("INSIGHTS & FINDINGS")
print("="*70)

print("\n📊 KEY FINDINGS:\n")

print(f"1. MOST INFLUENTIAL FEATURES:")
top_3_features = feature_importance.head(3)
for idx, (feat, imp) in enumerate(zip(top_3_features['Feature'], top_3_features['Importance']), 1):
    print(f"   {idx}. {feat} (Importance: {imp:.4f})")

print(f"\n2. MODEL PERFORMANCE:")
print(f"   - Best Model: {best_model_name}")
print(f"   - Test R² Score: {max(lr_test_r2, rf_test_r2):.4f}")
print(f"   - Test RMSE: ${min(lr_test_rmse, rf_test_rmse):,.2f}")
print(f"   - Test MAE: ${min(lr_test_mae, rf_test_mae):,.2f}")
print(f"   - Interpretation: The model explains {max(lr_test_r2, rf_test_r2)*100:.2f}% of price variance")

print(f"\n3. DATA INSIGHTS:")
print(f"   - Total samples: {len(df)}")
print(f"   - Total features: {X.shape[1]}")
print(f"   - Price range: ${y.min():,.0f} - ${y.max():,.0f}")
print(f"   - Average price: ${y.mean():,.0f}")

print(f"\n4. MODEL COMPARISON:")
print(f"   - Linear Regression R²: {lr_test_r2:.4f}")
print(f"   - Random Forest R²: {rf_test_r2:.4f}")
if abs(rf_test_r2 - lr_test_r2) > 0.05:
    print(f"   - Difference: {abs(rf_test_r2 - lr_test_r2):.4f} (Significant improvement with Random Forest)")
else:
    print(f"   - Both models perform similarly")

### Write Your Insights Here (5-8 lines)

**INSTRUCTIONS:** Replace the text below with your own insights based on the analysis above. Answer these questions:
1. Which features influence house price the most?
2. How accurate was your model (in plain terms)?
3. What surprised you in the data?
4. One recommendation for a real estate business based on your findings

---

**YOUR INSIGHTS:**

[WRITE YOUR 5-8 LINE SUMMARY HERE]

Example:
This analysis built two regression models (Linear Regression and Random Forest) to predict house prices based on 12 key property features. The Random Forest model achieved an R² score of 0.89, explaining 89% of price variance with a test RMSE of $45,000, indicating strong predictive power. The most influential features are [list top 3], accounting for the majority of price impact. Interestingly, [insert surprise], suggesting that real estate valuations are driven by [key insight]. For real estate businesses, this insight suggests focusing on [recommendation] to maximize property valuation and buyer attraction.


In [ ]:
# Summary for Export
summary_text = """
HOUSE PRICE PREDICTION - PROJECT SUMMARY
========================================

PROJECT OBJECTIVE:
Build a regression model to predict house prices and identify the most influential property features.

DATA SUMMARY:
- Total Samples: {}
- Features Used: {}
- Price Range: ${:,.0f} - ${:,.0f}
- Mean Price: ${:,.0f}

MODELS TRAINED:
1. Linear Regression
   - Test R² Score: {:.4f}
   - Test RMSE: ${:,.2f}
   - Test MAE: ${:,.2f}

2. Random Forest Regressor
   - Test R² Score: {:.4f}
   - Test RMSE: ${:,.2f}
   - Test MAE: ${:,.2f}

BEST MODEL: {} (R² = {:.4f})

TOP 3 INFLUENTIAL FEATURES:
{}

INTERPRETATION:
The {} model explains {:.2f}% of the price variance, demonstrating strong predictive capability.
The mean absolute error of ${:,.2f} suggests predictions are typically off by this amount.

DELIVERABLES:
✓ Jupyter Notebook with complete analysis
✓ 4 visualization charts (price distribution, correlation heatmap, actual vs predicted, feature importance)
✓ Data cleaning & preprocessing
✓ Model comparison & evaluation
✓ Insights & recommendations

""".format(
    len(df),
    X.shape[1],
    y.min(), y.max(),
    y.mean(),
    lr_test_r2, lr_test_rmse, lr_test_mae,
    rf_test_r2, rf_test_rmse, rf_test_mae,
    best_model_name, max(lr_test_r2, rf_test_r2),
    "\n".join([f"{i+1}. {feat}" for i, feat in enumerate(feature_importance['Feature'].head(3))]),
    best_model_name, max(lr_test_r2, rf_test_r2)*100,
    min(lr_test_mae, rf_test_mae)
)

print(summary_text)
print("\n✓ Summary generated. Copy this to your summary.docx or summary.pdf")

---

## Project Complete! ✅

### Next Steps for Submission:

1. **Edit Task 5:** Write your own insights (5-8 lines) in the markdown cell above

2. **Export Notebook:** Save this notebook as `analysis.ipynb`

3. **Create Summary Document:**
   - Convert this notebook to PDF/DOCX (File → Download as)
   - Or copy the summary text above into a Word document
   - Save as `summary.pdf` or `summary.docx`

4. **Organize Files:**
   ```
   HousePricePrediction_YourName/
   ├── analysis.ipynb
   ├── Housing.csv
   ├── summary.pdf (or .docx)
   └── charts/
       ├── chart_1_price_distribution.png
       ├── chart_2_correlation_heatmap.png
       ├── chart_3_actual_vs_predicted.png
       └── chart_4_feature_importance.png
   ```

5. **Compress & Submit:**
   - Zip the folder: `HousePricePrediction_YourName.zip`
   - Upload to the Google Form provided

---

**Good luck with your submission!** 🚀